In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import re

### conversion to pd df

In [12]:
df_users = pd.read_csv('DATA1/users.csv')
print(df_users.shape)
df_users.head(2)

(3293, 5)


,id,name,address,phone,email
0,44533,Hoyt Carter,"Apt. 300 8604 Ashlyn Wells, Effertzstad, ID 02997",(462) 385-4294,mckinley.rowe@harber.example
1,46128,Marco Kulas,"Apt. 538 816 Bechtelar Ferry, Lincolnhaven, KS...",913.466.4487,francisco@murray-cronin.test


In [13]:
df_orders = pd.read_parquet('DATA1/orders.parquet', engine='pyarrow')
print(df_orders.shape)
df_orders.head(2)

(11237, 7)


,id,user_id,book_id,quantity,unit_price,timestamp,shipping
0,71389,47288,18976,2,27.00$,10/01/24 10:38:08 A.M.,NaN
1,66343,47049,19403,1,€50¢50,10:14;19-Oct-2024,"4940 Arnoldo Keys, West Arnette, KS 77599"


In [14]:
with open('DATA1/books.yaml', 'r') as f:
    books_data = yaml.safe_load(f)

df_books = pd.DataFrame(books_data)
print(df_books.shape)
df_books.columns = df_books.columns.str.lstrip(':')

(753, 6)


In [6]:
df_books.head(2)

,id,title,author,genre,publisher,year
0,19199,The Yellow Meads of Asphodel,Carolyne West,Classic,Mainstream Publishing,2009
1,19398,From Here to Eternity,"Rep. Heath Stiedemann, Gino Welch, Haydee Larson",Short story,Vintage Books,2001


### nulls in data

In [7]:
users_total_nulls = df_users.isnull().sum().sum()
print(f"Total users missing values: {users_total_nulls}")

Total users missing values: 116


In [8]:
orders_total_nulls = df_orders.isnull().sum().sum()
print(f"Total orders missing values: {orders_total_nulls}")

Total orders missing values: 2810


In [9]:
books_total_nulls = df_books.isnull().sum().sum()
print(f"Total books missing values: {books_total_nulls}")

Total books missing values: 4


In [10]:
users_nulls = df_users.isnull().sum()
print("df_users: null columns:")
print(users_nulls[users_nulls > 0], "\n")

orders_nulls = df_orders.isnull().sum()
print("df_orders: null columns:")
print(orders_nulls[orders_nulls > 0], "\n")

books_nulls = df_books.isnull().sum()
print("df_books: null columns:")
print(books_nulls[books_nulls > 0], "\n")

df_users: null columns:
address    116
dtype: int64 

df_orders: null columns:
shipping    2810
dtype: int64 

df_books: null columns:
publisher    2
year         2
dtype: int64 



### feature processing & engineering

In [11]:
def clean_currency(price_str):
    if pd.isna(price_str):
        return price_str
    
    price_str = str(price_str)
    
    # Look for two groups of digits separated by any non-digit (e.g., "." or "¢")
    # This perfectly handles "27.00$" -> 27.00 and "€50¢50" -> 50.50
    match = re.search(r'(\d+)\D+(\d+)', price_str)
    if match:
        return float(f"{match.group(1)}.{match.group(2)}")
    
    # Fallback for simple whole numbers (e.g., "$15" -> 15.0)
    num_only = re.sub(r'[^\d.]', '', price_str)
    return float(num_only) if num_only else None